# Projeto: Pipeline de Churn Financeiro com Big Data (PySpark) e MLOps (MLflow)

**Objetivo:** Evoluir um modelo de predição de Churn (abandono de clientes) para lidar com alto volume de dados (Big Data).
Ao invés de utilizarmos o `pandas` (processamento em nó único), utilizaremos o **Apache Spark** (via PySpark) para permitir o processamento distribuído, garantindo que o algoritmo escale para milhões de registros financeiros sem travamentos. Além disso, utilizaremos o **MLflow** para o rastreamento do ciclo de vida do modelo.

### 1. Preparação do Ambiente e Infraestrutura
A primeira etapa é instalar os motores de processamento e governança na máquina virtual.


In [8]:
# Instalando o motor de Big Data (PySpark) e a plataforma de governança de ML (MLflow)
!pip install pyspark mlflow


### 2. Ligando o Motor do Cluster (Spark Session)
No mundo do Pandas, nós apenas importamos a biblioteca. No mundo do Big Data, nós precisamos iniciar uma **Spark Session**.

A Spark Session atua como o "Maestro" da orquestra: é ela quem recebe os seus comandos em Python e os traduz para a linguagem do motor do Spark (na JVM - Java Virtual Machine), distribuindo o processamento pela máquina.


In [9]:
from pyspark.sql import SparkSession

# Inicializando o Maestro (Cluster Local)
spark = SparkSession.builder \
    .appName("Churn_Financeiro_BigData") \
    .getOrCreate()

print("Motor do PySpark ligado com sucesso!")
print(f"Versão em execução: {spark.version}")


Motor do PySpark ligado com sucesso!
Versão em execução: 4.0.2


### 3. Ingestão de Dados (Zona Bronze)
Nossa primeira missão é carregar a base de clientes bruta.

A grande diferença estrutural aqui é: quando usamos `pd.read_csv()`, o Pandas lê o arquivo inteiro e o "enfia" goela abaixo na memória RAM.

Quando usamos `spark.read.csv()`, o Spark lê o arquivo, entende a estrutura dele e divide os dados em partições, sem estourar a memória.


In [10]:
# Lendo o arquivo carregado no Colab
# header=True indica que a primeira linha é o cabeçalho das colunas
# inferSchema=True pede para o Spark tentar adivinhar se a coluna é texto ou número
df_clientes = spark.read.csv("base_clientes.csv", header=True, inferSchema=True)

# No pandas usamos .head(), no Spark usamos .show()
print("Amostra dos Dados:")
df_clientes.show(5)

# No pandas usamos .info(), no Spark usamos .printSchema()
# O Schema é vital no Big Data para evitar travamentos de tipagem em tabelas gigantes
print("\nEsquema de Dados (Tipagem):")
df_clientes.printSchema()


Amostra dos Dados:
+----------+----------+-------------+------------+---------------+----------------+--------+-----+
|cliente_id|  segmento|meses_cliente|qtd_produtos|retorno_12m_pct|freq_contato_mes|saldo_bi|churn|
+----------+----------+-------------+------------+---------------+----------------+--------+-----+
|  CLI00000|    Varejo|          125|           6|           7.89|               4|  1.7262|    1|
|  CLI00001|    Wealth|            7|           6|           7.17|               7|  0.1693|    0|
|  CLI00002|Alta Renda|           29|           6|           9.69|               0|  0.0974|    0|
|  CLI00003|    Varejo|          139|           1|          11.86|               2|  0.5976|    0|
|  CLI00004|    Varejo|           33|           2|            5.7|               2|  0.1295|    0|
+----------+----------+-------------+------------+---------------+----------------+--------+-----+
only showing top 5 rows

Esquema de Dados (Tipagem):
root
 |-- cliente_id: string (nullabl

### 4. Análise Exploratória Distribuída (EDA)
Para manipularmos as colunas no PySpark, nós importamos um pacote de "funções".

Ele nos permite fazer agrupamentos, cálculos de média e contagens de forma muito semelhante ao SQL, mas usando Python.

Vamos investigar como o `churn` (cancelamento) se comporta dependendo do `segmento` do cliente.


In [11]:
from pyspark.sql.functions import col, mean, count, round

print("Taxa de Churn por Segmento:")

# Agrupando por segmento, contando os clientes e tirando a média do churn (que é 0 ou 1)
df_analise = df_clientes.groupBy("segmento") \
    .agg(
        count("cliente_id").alias("total_clientes"),
        round(mean("churn") * 100, 2).alias("taxa_churn_pct")
    ) \
    .orderBy(col("taxa_churn_pct").desc())

df_analise.show()


Taxa de Churn por Segmento:
+----------+--------------+--------------+
|  segmento|total_clientes|taxa_churn_pct|
+----------+--------------+--------------+
|    Varejo|           784|         25.64|
|Alta Renda|           270|         11.48|
| Corporate|            31|          6.45|
|    Wealth|           115|          5.22|
+----------+--------------+--------------+



### 5. Feature Engineering: O Jeito PySpark (MLlib)
A preparação de dados para Machine Learning no Spark é muito diferente do `Scikit-Learn`.

Enquanto no Pandas nós fazemos *One-Hot Encoding* e passamos um DataFrame cheio de colunas para o modelo, o Spark exige que **todas as variáveis preditivas (features) sejam agrupadas em uma única coluna do tipo "Vetor"**.

Para isso, usaremos duas ferramentas clássicas do Big Data:

1. **StringIndexer:** Transforma textos (ex: 'Varejo', 'Wealth') em números (0.0, 1.0).

2. **VectorAssembler:** Pega todas as colunas numéricas e as "esmaga" em uma única coluna chamada `features`.


In [6]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Transformando o texto do 'segmento' em números
indexer = StringIndexer(inputCol="segmento", outputCol="segmento_index")
# O .fit() entende as categorias e o .transform() aplica a mudança
df_prep = indexer.fit(df_clientes).transform(df_clientes)

# 2. Definindo quais colunas vamos usar para prever o Churn
colunas_features = [
    "segmento_index",
    "meses_cliente",
    "qtd_produtos",
    "retorno_12m_pct",
    "freq_contato_mes",
    "saldo_bi"
]

# 3. Juntando todas essas colunas em um único Vetor chamado 'features'
assembler = VectorAssembler(inputCols=colunas_features, outputCol="features")
df_final = assembler.transform(df_prep)

# Mostrando apenas a coluna target (churn) e o novo Vetor de features
print("Tabela preparada para o Robô de Machine Learning:")
df_final.select("cliente_id", "churn", "features").show(5, truncate=False)


Tabela preparada para o Robô de Machine Learning:
+----------+-----+--------------------------------+
|cliente_id|churn|features                        |
+----------+-----+--------------------------------+
|CLI00000  |1    |[0.0,125.0,6.0,7.89,4.0,1.7262] |
|CLI00001  |0    |[2.0,7.0,6.0,7.17,7.0,0.1693]   |
|CLI00002  |0    |[1.0,29.0,6.0,9.69,0.0,0.0974]  |
|CLI00003  |0    |[0.0,139.0,1.0,11.86,2.0,0.5976]|
|CLI00004  |0    |[0.0,33.0,2.0,5.7,2.0,0.1295]   |
+----------+-----+--------------------------------+
only showing top 5 rows


### 6. Rastreamento e Treinamento de Machine Learning (MLOps)
Em ambientes corporativos, você não treina um modelo apenas dando `.fit()`. Você precisa provar e rastrear qual foi o resultado.

Para isso utilizamos o **MLflow**. Ele "envolve" o nosso treinamento e anota (log) automaticamente quem treinou, quais foram os hiperparâmetros usados e as métricas resultantes, criando um histórico auditável do modelo.


In [7]:
import mlflow
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Separando os dados em Treino (70%) e Teste (30%) no PySpark
train_data, test_data = df_final.randomSplit([0.7, 0.3], seed=42)

print(f"Registros de Treino: {train_data.count()}")
print(f"Registros de Teste: {test_data.count()}")

# 2. Iniciando a gravação do experimento no MLflow
# O MLflow começa a gravar tudo que acontecer dentro deste bloco 'with'
with mlflow.start_run(run_name="RandomForest_Churn_V1"):

    # Configurando o Algoritmo (PySpark MLlib)
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="churn",
        numTrees=100,
        maxDepth=5
    )

    # Treinando o Robô
    print("\nTreinando o modelo de Random Forest em Cluster (PySpark)...")
    rf_model = rf.fit(train_data)

    # 3. Fazendo Previsões na base de Teste
    previsoes = rf_model.transform(test_data)

    # 4. Avaliando a qualidade do modelo (Acurácia / F1-Score)
    evaluator = MulticlassClassificationEvaluator(
        labelCol="churn",
        predictionCol="prediction",
        metricName="f1"
    )

    f1_score = evaluator.evaluate(previsoes)

    # 5. Salvando o registro no MLflow
    # Aqui o MLOps acontece: guardamos os hiperparâmetros e o resultado no cofre!
    mlflow.log_param("algoritmo", "Random Forest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_metric("f1_score", f1_score)

    print(f"\n[SUCESSO] Modelo Treinado! F1-Score do Teste: {f1_score:.4f}")
    print("Os dados do experimento foram salvos pelo MLflow de forma segura.")


Registros de Treino: 887
Registros de Teste: 313


2026/04/25 01:19:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/25 01:19:42 INFO mlflow.store.db.utils: Updating database tables



Treinando o modelo de Random Forest em Cluster (PySpark)...

[SUCESSO] Modelo Treinado! F1-Score do Teste: 0.6727
Os dados do experimento foram salvos pelo MLflow de forma segura.


### 7. Inspecionando o Resultado do Modelo
Vamos ver como a tabela sai do outro lado após passar pelo modelo.
O PySpark adiciona novas colunas no final com as probabilidades e a previsão final.


In [12]:
# Mostrando o cruzamento: o que era realidade (churn), as probabilidades do modelo, e o que o modelo previu
previsoes.select("cliente_id", "churn", "probability", "prediction").show(10, truncate=False)


+----------+-----+----------------------------------------+----------+
|cliente_id|churn|probability                             |prediction|
+----------+-----+----------------------------------------+----------+
|CLI00002  |0    |[0.7118972949800281,0.2881027050199719] |0.0       |
|CLI00006  |1    |[0.7914384499475424,0.20856155005245752]|0.0       |
|CLI00008  |0    |[0.7746045929987165,0.2253954070012835] |0.0       |
|CLI00009  |0    |[0.9036612772418671,0.09633872275813293]|0.0       |
|CLI00013  |0    |[0.779037557364391,0.22096244263560913] |0.0       |
|CLI00014  |1    |[0.72539426105987,0.27460573894012996]  |0.0       |
|CLI00015  |0    |[0.7651926562897018,0.2348073437102982] |0.0       |
|CLI00019  |1    |[0.8067594579144963,0.1932405420855036] |0.0       |
|CLI00021  |0    |[0.7494747982485443,0.25052520175145565]|0.0       |
|CLI00023  |1    |[0.7674214156957325,0.23257858430426753]|0.0       |
+----------+-----+----------------------------------------+----------+
only s

Para o cliente CLI00002 (que não cancelou, churn=0), a previsão foi 0.0. Certo!

Para o cliente CLI00006 (que cancelou, churn=1), a previsão também foi 0.0!

Se você olhar toda a coluna prediction, ele está chutando 0 para todo mundo.

Desbalanceamento de Classes

In [13]:
# Contando a proporção de Churn
df_clientes.groupBy("churn").count().show()


+-----+-----+
|churn|count|
+-----+-----+
|    1|  240|
|    0|  960|
+-----+-----+



### 8. Resolvendo o Desbalanceamento (Class Weights)
Como temos muito mais clientes fiéis do que clientes que dão churn, o algoritmo fica "preguiçoso" e chuta 0 para todos.

Vamos balancear isso matematicamente aplicando um "peso". Se temos 80% de clientes normais e 20% de churn, o peso do churn será muito maior para forçar a árvore de decisão a prestar atenção neles.


In [14]:
from pyspark.sql.functions import when

# 1. Calculando os pesos automaticamente (sem chutar números)
total_clientes = df_final.count()
qtd_churn = df_final.filter(col("churn") == 1).count()
qtd_fieis = total_clientes - qtd_churn

# A matemática do peso: quem tem menos, ganha peso maior
peso_churn = total_clientes / (2 * qtd_churn)
peso_fiel = total_clientes / (2 * qtd_fieis)

print(f"Peso aplicado para clientes Fiéis (0): {peso_fiel:.2f}")
print(f"Peso aplicado para clientes de Churn (1): {peso_churn:.2f}")

# 2. Criando uma nova coluna na base com o peso de cada cliente
df_balanceado = df_final.withColumn(
    "peso_classe",
    when(col("churn") == 1, peso_churn).otherwise(peso_fiel)
)

# 3. Refazendo a separação Treino/Teste
train_data_bal, test_data_bal = df_balanceado.randomSplit([0.7, 0.3], seed=42)


Peso aplicado para clientes Fiéis (0): 0.62
Peso aplicado para clientes de Churn (1): 2.50


### 9. Treinando o Modelo V2 (Balanceado) no MLflow
Agora vamos rodar o modelo novamente, mas desta vez passaremos o parâmetro `weightCol` para o Random Forest, dizendo a ele para respeitar a punição imposta pela nossa nova coluna.


In [15]:
# Iniciando uma NOVA gravação no MLflow para a Versão 2
with mlflow.start_run(run_name="RandomForest_Churn_V2_Balanceado"):

    # Repare no novo parâmetro: weightCol="peso_classe"
    rf_v2 = RandomForestClassifier(
        featuresCol="features",
        labelCol="churn",
        weightCol="peso_classe", # A MÁGICA ACONTECE AQUI
        numTrees=100,
        maxDepth=5
    )

    print("Treinando o modelo V2 Balanceado...")
    rf_model_v2 = rf_v2.fit(train_data_bal)

    previsoes_v2 = rf_model_v2.transform(test_data_bal)

    # Avaliando
    f1_score_v2 = evaluator.evaluate(previsoes_v2)

    # Salvando no MLflow
    mlflow.log_param("algoritmo", "Random Forest")
    mlflow.log_param("balanceamento", "Class Weights")
    mlflow.log_metric("f1_score", f1_score_v2)

    print(f"\n[SUCESSO] Modelo V2 Treinado! Novo F1-Score: {f1_score_v2:.4f}")

# Mostrando o resultado lado a lado novamente
print("\nNovas Previsões (Observe se agora aparecem previsões '1.0'):")
previsoes_v2.select("cliente_id", "churn", "probability", "prediction").show(10, truncate=False)


Treinando o modelo V2 Balanceado...

[SUCESSO] Modelo V2 Treinado! Novo F1-Score: 0.7083

Novas Previsões (Observe se agora aparecem previsões '1.0'):
+----------+-----+----------------------------------------+----------+
|cliente_id|churn|probability                             |prediction|
+----------+-----+----------------------------------------+----------+
|CLI00002  |0    |[0.5530639050004299,0.44693609499957015]|0.0       |
|CLI00006  |1    |[0.4818801778429374,0.5181198221570626] |1.0       |
|CLI00008  |0    |[0.4236348260262152,0.5763651739737847] |1.0       |
|CLI00009  |0    |[0.7875720052141291,0.21242799478587085]|0.0       |
|CLI00013  |0    |[0.5845447906671911,0.4154552093328089] |0.0       |
|CLI00014  |1    |[0.37715994831039706,0.6228400516896029]|1.0       |
|CLI00015  |0    |[0.5480009973302519,0.45199900266974813]|0.0       |
|CLI00019  |1    |[0.5290155663864489,0.47098443361355113]|0.0       |
|CLI00021  |0    |[0.49807755326437847,0.5019224467356215]|1.0      

A Cura da Cegueira: O modelo deixou de chutar "0" para todo mundo.

Veja o cliente CLI00006, o CLI00014 e o CLI00023! A coluna churn (realidade) diz que eles saíram (1), e a nossa coluna prediction apontou 1.0. O algoritmo aprendeu a caçar o perfil de cancelamento!

O Salto do F1-Score: Um F1-Score de 0.70 em uma base de dados crua e apenas com os pesos balanceados é um resultado formidável para um primeiro ciclo de Big Data.

Falso Negativo (Deixar um cliente fugir sem ver): Custa milhares de reais de perda de receita contínua.

Falso Positivo (Achar que alguém ia fugir, mas não ia): O máximo que o banco vai fazer é mandar um "brinde" ou uma oferta para ele ficar.

O custo disso é infinitamente menor do que perder o cliente. Os pesos de classe forçaram a IA a ter mais "medo" de perder cliente do que de dar desconto.